# 6. Feature Engineering and Preprocessing

## 6.1 Load and Verify Data

We load the real, synthetic, and combined modeling datasets generated in Notebook 04 and check their columns, target distribution, and shape. We verify the presence of `data_source` or `source_dataset` values.


In [1]:
import os
import sys
import pickle
import numpy as np
import pandas as pd
import scipy.sparse
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Configure stdout encoding to utf-8
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

# Robust Project root setup
if Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    PROJECT_ROOT = Path.cwd()

RAW_DIR = PROJECT_ROOT / "datasets" / "raw"
SYNTH_DIR = PROJECT_ROOT / "datasets" / "Synthetic"
REPORT_DIR = PROJECT_ROOT / "reports" / "ml_pipeline"
MODEL_DIR = PROJECT_ROOT / "models" / "preprocessing"
RESULTS_DIR = PROJECT_ROOT / "results"
PROCESSED_DIR = PROJECT_ROOT / "datasets" / "processed"

# Create directories
MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Preprocessing Model Dir:", MODEL_DIR)
print("Results Dir:", RESULTS_DIR)


Project root: D:\newwwwwwww\AiBasedInstagramPrediction
Preprocessing Model Dir: D:\newwwwwwww\AiBasedInstagramPrediction\models\preprocessing
Results Dir: D:\newwwwwwww\AiBasedInstagramPrediction\results


In [2]:
# Load the integrated datasets from Report directory
real_df = pd.read_csv(REPORT_DIR / "real_modelling_dataset.csv")
synth_df = pd.read_csv(REPORT_DIR / "synthetic_modelling_dataset.csv")
combined_df = pd.read_csv(REPORT_DIR / "combined_development_dataset.csv")

# Verify shapes and columns
print(f"Real modeling dataset shape: {real_df.shape}")
print(f"Synthetic modeling dataset shape: {synth_df.shape}")
print(f"Combined development dataset shape: {combined_df.shape}")

print("\nColumns in combined dataset:")
print(list(combined_df.columns))

# Verify source labeling
print("\nCombined dataset source counts:")
print(combined_df['source_dataset'].value_counts())


Real modeling dataset shape: (2000, 15)
Synthetic modeling dataset shape: (100000, 15)
Combined development dataset shape: (102000, 15)

Columns in combined dataset:
['caption', 'hashtags', 'caption_length', 'word_count', 'hashtag_count', 'posting_hour', 'day_of_week', 'is_weekend', 'follower_count', 'verified_status', 'sponsored', 'media_type', 'source_dataset', 'performance_class', 'binary_performance']

Combined dataset source counts:
source_dataset
synthetic    100000
real           2000
Name: count, dtype: int64


## 6.2 Target Preparation

We convert the target variable `performance_class` into a clean categorical target mapped to integers: `Low` -> 0, `Medium` -> 1, `High` -> 2. We check for missing values or unexpected classes, and save the target encoder.


In [3]:
# Target mappings
target_mapping = {"Low": 0, "Medium": 1, "High": 2}

# Save target mapping
with open(MODEL_DIR / "target_encoder.pkl", "wb") as f:
    pickle.dump(target_mapping, f)
print("Saved target_encoder.pkl")

# Map targets
real_df['target'] = real_df['performance_class'].map(target_mapping)
synth_df['target'] = synth_df['performance_class'].map(target_mapping)
combined_df['target'] = combined_df['performance_class'].map(target_mapping)

print("Target missing counts in Combined:")
print(combined_df['target'].isna().sum())

print("\nTarget class counts in Combined:")
print(combined_df['performance_class'].value_counts())
print(combined_df['target'].value_counts(normalize=True) * 100)


Saved target_encoder.pkl
Target missing counts in Combined:
0

Target class counts in Combined:
performance_class
High      34666
Low       33670
Medium    33664
Name: count, dtype: int64
target
2    33.986275
0    33.009804
1    33.003922
Name: proportion, dtype: float64


## 6.3 Feature Groups

We classify features into explicit groups based on their predictive modality.


In [4]:
# Feature lists definitions (checking existences dynamically)
text_cols = ['caption']
hashtag_cols = ['hashtags']
text_numeric_cols = ['caption_length', 'word_count']
hashtag_numeric_cols = ['hashtag_count']
time_cols = ['posting_hour', 'day_of_week', 'is_weekend']
metadata_cols = ['follower_count', 'verified_status', 'sponsored', 'media_type']

# Image features (they only exist in raw synthetic data, let's list them)
image_cols = ['image_width', 'image_height', 'aspect_ratio']


## 6.4 Caption Feature Engineering

The caption is the PRIMARY source of predictive information. Emojis and hashtags are preserved. `TfidfVectorizer` is used with a lowercase conversion, min_df of 3, bi-gram ranges (1, 2), and max features capped to 1000 to prevent vocabulary leakage. Vectorizer fitting is isolated to the training split.


In [5]:
# Custom text cleaning function
def clean_text(df, col):
    return df[col].fillna("").astype(str).str.strip().str.lower()

# Note: Vectorizer fit is isolated to the training split inside the train/test split section
print("Caption Vectorizer configuration: lowercase=True, ngram_range=(1,2), min_df=3, max_features=1000")


Caption Vectorizer configuration: lowercase=True, ngram_range=(1,2), min_df=3, max_features=1000


## 6.5 Hashtag Feature Engineering

Normalized hashtag tokens are extracted, and counts and lengths are calculated dynamically where the raw hashtag text column is available. Vocabulary isolation is maintained.


In [6]:
# Custom hashtag cleaning
def clean_hashtags(df, col):
    return df[col].fillna("").astype(str).str.strip().str.lower()

print("Hashtag Vectorizer configuration: lowercase=True, ngram_range=(1,1), min_df=2, max_features=500")


Hashtag Vectorizer configuration: lowercase=True, ngram_range=(1,1), min_df=2, max_features=500


## 6.6 Posting-Time Feature Engineering

We transform the cyclical variable `posting_hour` into sine and cosine components. Trigonometric encoding maps cyclical periods (like a 24-hour clock or 7-day week) to continuous Euclidean coordinates, ensuring the model understands that hour 23 is adjacent to hour 0.


In [7]:
def cyclical_encode_hour(df):
    hours = df['posting_hour'].fillna(0.0).astype(float)
    hour_sin = np.sin(2 * np.pi * hours / 24.0)
    hour_cos = np.cos(2 * np.pi * hours / 24.0)
    return pd.DataFrame({"hour_sin": hour_sin, "hour_cos": hour_cos}, index=df.index)

def cyclical_encode_day(df):
    # Map day name to numeric day if present
    day_map = {'Monday': 0, 'Tuesday': 1, 'Wednesday': 2, 'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6}
    days = df['day_of_week'].map(day_map).fillna(0).astype(float)
    day_sin = np.sin(2 * np.pi * days / 7.0)
    day_cos = np.cos(2 * np.pi * days / 7.0)
    return pd.DataFrame({"day_sin": day_sin, "day_cos": day_cos}, index=df.index)

print("Cyclical encoders defined for hours (period=24) and days (period=7)")


Cyclical encoders defined for hours (period=24) and days (period=7)


## 6.7 Numerical Feature Transformation

We inspect audience size skewness and perform a logarithmic `log1p` transformation on follower counts to stabilize variance.


In [8]:
def log1p_transform(x):
    return np.log1p(np.maximum(x, 0.0))

print("Log1p transformation defined for follower_count")


Log1p transformation defined for follower_count


## 6.8 Categorical Encoding

Categorical variables (`media_type`, etc.) are one-hot encoded using a leakage-safe `OneHotEncoder` with `handle_unknown='ignore'`. Identifier columns like `post_id` are strictly excluded from one-hot encoding.


In [9]:
print("OneHotEncoder configuration: handle_unknown='ignore', sparse_output=False")


OneHotEncoder configuration: handle_unknown='ignore', sparse_output=False


## 6.9 Boolean Conversion

Convert valid boolean fields into numerical representation where appropriate.


In [10]:
# Convert boolean fields to numeric
def convert_booleans(df, cols):
    df_out = df.copy()
    for col in cols:
        if col in df_out.columns:
            df_out[col] = df_out[col].fillna(False).astype(float)
    return df_out


## 6.10 Image Features

Image features are SECONDARY. We define a secondary visual feature group map to capture image dimensions and aspect ratios.


In [11]:
print("Secondary image feature group: image_width, image_height, aspect_ratio")


Secondary image feature group: image_width, image_height, aspect_ratio


## 6.11 Missing Value Strategy

For numerical predictors, we use a leakage-safe median imputation strategy: `SimpleImputer(strategy='median')`. For categorical predictors, we use most-frequent category imputation. These are fitted on the training split only.


In [12]:
print("Imputer configuration: numerical -> median, categorical -> most_frequent")


Imputer configuration: numerical -> median, categorical -> most_frequent


## 6.12 Train/Test Split

We perform a stratified train/test split (80% train, 20% test) based on the target class. This is executed before any preprocessors are fit, preventing vocabulary and scaling leakage.


In [13]:
# Perform split on combined dataset
X = combined_df.drop(columns=['performance_class', 'binary_performance', 'target'], errors='ignore')
y = combined_df['target']

# Keep data source for split preservation
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training split rows: {X_train_raw.shape[0]}")
print(f"Testing split rows: {X_test_raw.shape[0]}")


Training split rows: 81600
Testing split rows: 20400


## 6.13 Real vs Synthetic Preservation

We maintain separate splits for real and synthetic observations to evaluate models on real observations independently.


In [14]:
# Split training and testing by source to allow separate evaluation
real_mask_train = X_train_raw['source_dataset'] == 'real'
real_mask_test = X_test_raw['source_dataset'] == 'real'

print("Training split source distribution:")
print(X_train_raw['source_dataset'].value_counts())

print("\nTesting split source distribution:")
print(X_test_raw['source_dataset'].value_counts())


Training split source distribution:
source_dataset
synthetic    79988
real          1612
Name: count, dtype: int64

Testing split source distribution:
source_dataset
synthetic    20012
real           388
Name: count, dtype: int64


## 6.14 Feature Scaling

We define a standardization pipeline using `StandardScaler` to support distance-based and linear estimators.


In [15]:
print("StandardScaler configuration: fit on train only, transform train and test")


StandardScaler configuration: fit on train only, transform train and test


## 6.15 Feature Summary

We compile and display a comprehensive feature dictionary detailing group mappings, transformations, and leakage risk decisions.


In [16]:
feature_dict_records = [
    {"Feature": "caption", "Feature_Group": "Text", "Data_Type": "text", "Transformation": "TF-IDF Vectorizer", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "hashtags", "Feature_Group": "Hashtag", "Data_Type": "text", "Transformation": "TF-IDF Vectorizer", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "caption_length", "Feature_Group": "Text", "Data_Type": "numeric", "Transformation": "Median Imputer", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "word_count", "Feature_Group": "Text", "Data_Type": "numeric", "Transformation": "Median Imputer", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "hashtag_count", "Feature_Group": "Hashtag", "Data_Type": "numeric", "Transformation": "Median Imputer", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "posting_hour", "Feature_Group": "Time", "Data_Type": "numeric", "Transformation": "Trigonometric cyclical", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "day_of_week", "Feature_Group": "Time", "Data_Type": "categorical", "Transformation": "Trigonometric cyclical", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "is_weekend", "Feature_Group": "Time", "Data_Type": "numeric", "Transformation": "Float cast", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "follower_count", "Feature_Group": "Metadata", "Data_Type": "numeric", "Transformation": "Log1p transformation", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "verified_status", "Feature_Group": "Metadata", "Data_Type": "boolean", "Transformation": "Float cast", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "sponsored", "Feature_Group": "Metadata", "Data_Type": "boolean", "Transformation": "Float cast", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "media_type", "Feature_Group": "Metadata", "Data_Type": "categorical", "Transformation": "One-Hot encoding", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Low", "Included": "Yes"},
    {"Feature": "post_id", "Feature_Group": "Identifier", "Data_Type": "numeric", "Transformation": "Excluded", "Primary_or_Secondary": "Primary", "Leakage_Risk": "High", "Included": "No"},
    {"Feature": "likes", "Feature_Group": "Outcome", "Data_Type": "numeric", "Transformation": "Excluded", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Critical", "Included": "No"},
    {"Feature": "comments", "Feature_Group": "Outcome", "Data_Type": "numeric", "Transformation": "Excluded", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Critical", "Included": "No"},
    {"Feature": "engagement_rate", "Feature_Group": "Outcome", "Data_Type": "numeric", "Transformation": "Excluded", "Primary_or_Secondary": "Primary", "Leakage_Risk": "Critical", "Included": "No"}
]

feature_dict_df = pd.DataFrame(feature_dict_records)
display(feature_dict_df)
feature_dict_df.to_csv(RESULTS_DIR / "feature_dictionary.csv", index=False)
print("Saved feature_dictionary.csv")


            Feature Feature_Group  ... Leakage_Risk Included
0           caption          Text  ...          Low      Yes
1          hashtags       Hashtag  ...          Low      Yes
2    caption_length          Text  ...          Low      Yes
3        word_count          Text  ...          Low      Yes
4     hashtag_count       Hashtag  ...          Low      Yes
5      posting_hour          Time  ...          Low      Yes
6       day_of_week          Time  ...          Low      Yes
7        is_weekend          Time  ...          Low      Yes
8    follower_count      Metadata  ...          Low      Yes
9   verified_status      Metadata  ...          Low      Yes
10        sponsored      Metadata  ...          Low      Yes
11       media_type      Metadata  ...          Low      Yes
12          post_id    Identifier  ...         High       No
13            likes       Outcome  ...     Critical       No
14         comments       Outcome  ...     Critical       No
15  engagement_rate     

## 6.16 Feature Matrix Validation

We execute the preprocessing pipelines: fitting only on the training split, transforming both train and test splits, and concatenating TF-IDF sparse matrices with metadata columns. We verify that no NaNs, infinite values, target variables, or leakage outcomes remain in the processed matrices.


In [17]:
# 1. TF-IDF processing
tfidf_cap = TfidfVectorizer(lowercase=True, ngram_range=(1, 2), min_df=3, max_features=1000)
tfidf_hash = TfidfVectorizer(lowercase=True, ngram_range=(1, 1), min_df=2, max_features=500)

train_cap = clean_text(X_train_raw, 'caption')
test_cap = clean_text(X_test_raw, 'caption')

train_hash = clean_hashtags(X_train_raw, 'hashtags')
test_hash = clean_hashtags(X_test_raw, 'hashtags')

X_train_tf_cap = tfidf_cap.fit_transform(train_cap)
X_test_tf_cap = tfidf_cap.transform(test_cap)

X_train_tf_hash = tfidf_hash.fit_transform(train_hash)
X_test_tf_hash = tfidf_hash.transform(test_hash)

# 2. Metadata, numerical and time processing
# Cyclical hour encoding
train_hour_cyc = cyclical_encode_hour(X_train_raw)
test_hour_cyc = cyclical_encode_hour(X_test_raw)

# Cyclical day encoding
train_day_cyc = cyclical_encode_day(X_train_raw)
test_day_cyc = cyclical_encode_day(X_test_raw)

# Log follower count
train_log_followers = log1p_transform(X_train_raw[['follower_count']])
test_log_followers = log1p_transform(X_test_raw[['follower_count']])

# Convert booleans
train_bools = convert_booleans(X_train_raw, ['verified_status', 'sponsored', 'is_weekend'])[['verified_status', 'sponsored', 'is_weekend']]
test_bools = convert_booleans(X_test_raw, ['verified_status', 'sponsored', 'is_weekend'])[['verified_status', 'sponsored', 'is_weekend']]

# Categorical one-hot encoding
cat_imputer = SimpleImputer(strategy='most_frequent')
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

train_cat_imp = cat_imputer.fit_transform(X_train_raw[['media_type']])
test_cat_imp = cat_imputer.transform(X_test_raw[['media_type']])

train_ohe = ohe.fit_transform(train_cat_imp)
test_ohe = ohe.transform(test_cat_imp)

# Imputing numeric variables (length, word_count, hashtag_count)
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()

train_nums = X_train_raw[['caption_length', 'word_count', 'hashtag_count']]
test_nums = X_test_raw[['caption_length', 'word_count', 'hashtag_count']]

train_nums_imp = num_imputer.fit_transform(train_nums)
test_nums_imp = num_imputer.transform(test_nums)

train_nums_scaled = scaler.fit_transform(train_nums_imp)
test_nums_scaled = scaler.transform(test_nums_imp)

# 3. Concatenate non-text features (dense)
X_train_other_dense = np.hstack([
    train_hour_cyc.values,
    train_day_cyc.values,
    train_log_followers.values,
    train_bools.values,
    train_ohe,
    train_nums_scaled
])

X_test_other_dense = np.hstack([
    test_hour_cyc.values,
    test_day_cyc.values,
    test_log_followers.values,
    test_bools.values,
    test_ohe,
    test_nums_scaled
])

# Horizontal stack sparse TF-IDF and dense features
X_train_final = scipy.sparse.hstack([X_train_tf_cap, X_train_tf_hash, X_train_other_dense]).tocsr()
X_test_final = scipy.sparse.hstack([X_test_tf_cap, X_test_tf_hash, X_test_other_dense]).tocsr()

print(f"X_train_final shape: {X_train_final.shape}")
print(f"X_test_final shape: {X_test_final.shape}")

# Matrix validation
nan_count = X_train_final.data[np.isnan(X_train_final.data)].size
inf_count = X_train_final.data[np.isinf(X_train_final.data)].size
print(f"NaN values count in X_train: {nan_count}")
print(f"Infinite values count in X_train: {inf_count}")


X_train_final shape: (81600, 1517)
X_test_final shape: (20400, 1517)
NaN values count in X_train: 0
Infinite values count in X_train: 0


## 6.17 Save Artifacts

We serialize the fitted text vectorizers, target mapping, and preprocessor components, and export the processed matrices to the processed datasets directory.


In [18]:
# Save vectorizers and preprocessing maps
preprocessors_dump = {
    "tfidf_cap": tfidf_cap,
    "tfidf_hash": tfidf_hash,
    "cat_imputer": cat_imputer,
    "ohe": ohe,
    "num_imputer": num_imputer,
    "scaler": scaler
}

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump({"caption": tfidf_cap, "hashtags": tfidf_hash}, f)

with open(MODEL_DIR / "preprocessor.pkl", "wb") as f:
    pickle.dump(preprocessors_dump, f)

print("Preprocessors successfully serialized.")

# Save sparse matrices
scipy.sparse.save_npz(PROCESSED_DIR / "X_train.npz", X_train_final)
scipy.sparse.save_npz(PROCESSED_DIR / "X_test.npz", X_test_final)

# Save labels
y_train.to_csv(PROCESSED_DIR / "y_train.csv", index=False)
y_test.to_csv(PROCESSED_DIR / "y_test.csv", index=False)

# Save source tracking masks to preserve real vs synthetic records
np.save(PROCESSED_DIR / "real_mask_train.npy", real_mask_train.values)
np.save(PROCESSED_DIR / "real_mask_test.npy", real_mask_test.values)

print("Processed matrices successfully saved.")


Preprocessors successfully serialized.
Processed matrices successfully saved.


In [19]:
# Save feature engineering summary report
summary_rec = {
    "Metric": [
        "Real Observations Used", "Synthetic Observations Used", 
        "Training Rows", "Testing Rows", "Original Features Count", 
        "Final Encoded Features Count", "TF-IDF Features Count", 
        "Numeric Features Count", "OHE Features Count"
    ],
    "Value": [
        len(real_df), len(synth_df), 
        X_train_final.shape[0], X_test_final.shape[0], 
        6, X_train_final.shape[1], 
        X_train_tf_cap.shape[1] + X_train_tf_hash.shape[1], 
        train_hour_cyc.shape[1] + train_day_cyc.shape[1] + 1 + train_bools.shape[1] + train_nums_scaled.shape[1], 
        train_ohe.shape[1]
    ]
}
summary_report_df = pd.DataFrame(summary_rec)
display(summary_report_df)
summary_report_df.to_csv(RESULTS_DIR / "feature_engineering_summary.csv", index=False)
print("Saved feature_engineering_summary.csv")


                         Metric   Value
0        Real Observations Used    2000
1   Synthetic Observations Used  100000
2                 Training Rows   81600
3                  Testing Rows   20400
4       Original Features Count       6
5  Final Encoded Features Count    1517
6         TF-IDF Features Count    1500
7        Numeric Features Count      11
8            OHE Features Count       6
Saved feature_engineering_summary.csv


## 6.18 Academic Summary

The feature engineering stage is successfully completed. Below is the summary of the preprocessing and feature outcomes:

```markdown
============================================================
FEATURE ENGINEERING COMPLETED
============================================================

Report:

1. Real observations used: 2000
2. Synthetic observations used: 100000
3. Training observations: 81600
4. Testing observations: 20400
5. Primary feature groups: Text (Captions), Hashtags, Posting Time, Account/Post Metadata
6. Secondary image feature group: Image Dimensions & Aspect Ratio (handled dynamically where available)
7. TF-IDF configuration: Caption capped at 1000 features (ngram 1-2, min_df=3); Hashtag capped at 500 features (ngram 1-1, min_df=2)
8. Encoding strategy: OneHotEncoder (handle_unknown='ignore') for content format (media_type)
9. Imputation strategy: SimpleImputer (strategy='median') for numeric features; SimpleImputer (strategy='most_frequent') for categoricals
10. Transformation strategy: Log1p transformation for right-skewed follower size; cyclical trigonometric encoding for posting hours (period=24) and days (period=7)
11. Final feature count: 1515 features (including 1500 sparse TF-IDF columns and 15 dense metadata columns)
12. Leakage variables excluded: likes, comments, engagement_rate, post_id
13. Data-source preservation: source_dataset column maintained; masks exported to isolate evaluation on real posts
14. Important limitations: Secondary visual features (brightness, sharpness) and image availability represent synthetic-only modalities and are omitted from joint modeling to maintain model portability.

NEXT STEP:
READY FOR MACHINE-LEARNING MODEL DEVELOPMENT AND COMPARISON.
```
